In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel
from kobert_tokenizer import KoBERTTokenizer

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [6]:
class BERTClassifier(nn.Module):
    def __init__(self, bert, hidden_size=768, num_classes=2, dr_rate=0.5):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(p=dr_rate)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled_output = outputs.pooler_output
        out = self.dropout(pooled_output)
        return self.classifier(out)

In [7]:
bertmodel = BertModel.from_pretrained("skt/kobert-base-v1")
tokenizer = KoBERTTokenizer.from_pretrained("skt/kobert-base-v1")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


### 원본 데이터로 학습한 모델 예측

In [ ]:
# 분류 모델 생성
model = BERTClassifier(bertmodel)
model.load_state_dict(torch.load(
    "../model/kobert_raw.pt",
    map_location=device
))
model.to(device)
model.eval()

In [ ]:
def predict(predict_sentence):
    if not isinstance(predict_sentence, str) or predict_sentence in ["등록된 내용이 없습니다.", "등록된 결과가 없습니다."] or pd.isna(predict_sentence):
        return "unclassifiable"

    # 인코딩
    encoding = tokenizer(
        predict_sentence,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    token_type_ids = encoding['token_type_ids'].to(device)

    # 예측
    with torch.no_grad():
        outputs = model(input_ids, attention_mask, token_type_ids)
        logits = outputs.detach().cpu().numpy()

    if np.argmax(logits) == 0:
        return "부정적"
    else:
        return "긍정적"


In [ ]:
def sentiment_to_number(sentiment):
    if sentiment == "긍정적":
        return 1
    elif sentiment == "부정적":
        return -1
    else:
        return 0


In [ ]:
Q_data = pd.read_csv('../data/Q_review.csv')

# 예측 수행
Q_data['AI_HUB_PT_predict'] = Q_data['리뷰상세내용'].apply(lambda x: sentiment_to_number(predict(x)))

# 결과 확인
print(Q_data[['리뷰상세내용', 'AI_HUB_PT_predict']].head())

# 결과 저장
Q_data.to_csv('../results/kobert_raw_predictions.csv', index=False)


### 언더샘플링 처리한 데이터로 학습한 모델 예측

In [ ]:
# 분류 모델 생성
model = BERTClassifier(bertmodel)
model.load_state_dict(torch.load(
    "../model/kobert_under.pt",
    map_location=device
))
model.to(device)
model.eval()

In [ ]:
Q_data = pd.read_csv('../data/Q_review.csv')

# 예측 수행
Q_data['kobert_undersampled_predict'] = Q_data['리뷰상세내용'].apply(lambda x: sentiment_to_number(predict(x)))

# 결과 확인
print(Q_data[['리뷰상세내용', 'kobert_undersampled_predict']].head())

# 결과 저장
Q_data.to_csv('../results/kobert_under_predictions.csv', index=False)
